# PRAGMA · A‑E(−1) v1.3 — diagnóstico prerregistrado del caso chica · un clic

> **No tienes que decidir ni mirar nada.** Arrastra la foto `P1070614.JPG` al panel **Archivos**
> (icono de carpeta, a la izquierda) y pulsa **Entorno de ejecución → Ejecutar todas**. Al final se
> descarga un ZIP: **adjúntalo solo a Claude**.

Qué hace, sin intervención:

1. Instala SAM 2 **en el commit exacto** `2b90b9f5` y descarga SAM 2.1 Large. **Se detiene** si el
   commit o el checkpoint no son los congelados.
2. Encuentra la foto por su huella SHA‑256 y **se detiene** si la luma de cualquiera de los prompts
   se desvía más de 3,0 de lo registrado.
3. Ejecuta las **178 llamadas** del prerregistro: BASE, +POS_HAIR, +POS_SLEEVE, +POS_HAIR+SLEEVE,
   recíproco y perturbaciones.
4. Empaqueta las 210 máscaras con un manifiesto de hashes en
   `PRAGMA_AEM1v13_<run>_PENDING_EXTERNAL_AUDIT.zip`.

**Por qué no enseña resultados:** la auditoría es **ciega y a doble llave**. Si Colab mostrara las
máscaras, alguien podría ver o contar un resultado antes de que las dos IAs juzguen a ciegas. No
compartas capturas de este cuaderno con ChatGPT ni con nadie: el paquete ciego lo prepara Claude.

Prerregistro: `aem1/PRERREGISTRO_A-E-menos-1_v1_3.json` (SHA‑256 del archivo `9953ed9cb22e1b54a2a7116450158f8d68d48269ec679b3e53309d839ff29fb8`;
`content_sha256` `5800f2bf624620c353067210c73d782a8aefa39563925e859929ef0cba432a8c`). Auditoría: `auditoria/PROTOCOLO_AUDITORIA_AEM1_v2.md` (rev. 1).

## 0. Antes de pulsar «Ejecutar todas»

1. **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU** (L4 o A100; T4 también
   sirve, pero es más lenta). La corrida 1 usó L4.
2. Arrastra `P1070614.JPG` al panel **Archivos**. Si no lo haces, la celda 2 te pedirá la foto con
   un botón **Elegir archivos**. El nombre no importa: se reconoce por su huella.
3. **Entorno de ejecución → Ejecutar todas.** Tarda unos minutos: la mayor parte es instalar SAM 2
   y descargar 857 MiB. Al terminar, el navegador descarga el ZIP.

Si aparece un error en rojo, copia el texto y pégalo a Claude. No cambies nada del cuaderno.

In [ ]:
# Celda 1 · SAM 2 en el commit congelado y checkpoint verificado (red; en el arnés local se omite)
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

SAM2_PINNED_COMMIT = "2b90b9f5ceec907a1c18123530e92e794ad901a4"
CHECKPOINT_SHA256_PINNED = "2647878d5dfa5098f2f8649825738a9345572bae2d4350a2468587ece47dd318"
CHECKPOINT_BYTES_PINNED = 898083611
REPO_DIR = Path("/content/pragma_sam2_2b90b9f5")
WORK_DIR = Path("/content/pragma_run")
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
CHECKPOINT = CHECKPOINT_DIR / "sam2.1_hiera_large.pt"
CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt"
MODEL_CFG = "configs/sam2.1/sam2.1_hiera_l.yaml"
WORK_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def run_checked(args, **kwargs):
    print("$", " ".join(map(str, args)))
    return subprocess.run(args, check=True, text=True, **kwargs)

if not (REPO_DIR / ".git").exists():
    # Clon sin blobs y checkout del commit exacto (no «main», que puede moverse).
    run_checked(["git", "clone", "--filter=blob:none", "--no-checkout",
                 "https://github.com/facebookresearch/sam2.git", str(REPO_DIR)])
run_checked(["git", "-C", str(REPO_DIR), "checkout", "--quiet", SAM2_PINNED_COMMIT])
SAM2_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if SAM2_COMMIT != SAM2_PINNED_COMMIT:
    raise RuntimeError(f"FAIL_FREEZE: SAM 2 está en {SAM2_COMMIT}, no en {SAM2_PINNED_COMMIT}. No se genera nada.")

install_env = os.environ.copy()
install_env["SAM2_BUILD_CUDA"] = "0"
run_checked([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO_DIR, env=install_env)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
import importlib
importlib.invalidate_caches()
import sam2
print("SAM 2 importable:", Path(sam2.__file__).resolve())

def download(url, destination):
    temporary = destination.with_suffix(destination.suffix + ".part")
    def progress(blocks, block_size, total):
        if total > 0 and blocks % 128 == 0:
            print(f"Descarga: {min(100, blocks*block_size*100/total):5.1f}%", end="\r")
    urllib.request.urlretrieve(url, temporary, reporthook=progress)
    temporary.replace(destination)
    print(f"\nCheckpoint: {destination.stat().st_size / 2**20:.1f} MiB")

if not CHECKPOINT.exists() or CHECKPOINT.stat().st_size != CHECKPOINT_BYTES_PINNED:
    download(CHECKPOINT_URL, CHECKPOINT)
digest = hashlib.sha256()
with open(CHECKPOINT, "rb") as handle:
    for block in iter(lambda: handle.read(1 << 20), b""):
        digest.update(block)
if CHECKPOINT.stat().st_size != CHECKPOINT_BYTES_PINNED or digest.hexdigest() != CHECKPOINT_SHA256_PINNED:
    raise RuntimeError("FAIL_FREEZE: el checkpoint no es el congelado (bytes o SHA-256). No se genera nada.")
os.chdir(WORK_DIR)
print("Congelado verificado · SAM 2", SAM2_COMMIT, "· checkpoint", CHECKPOINT_SHA256_PINNED[:12], "…")

In [ ]:
# Celda 2 · hardware, precisión y foto (se reconoce por SHA-256; el nombre puede variar)
EXPECTED_IMAGE_SHA256 = "8f6e3b6f5013265a45c7e89121e3f0a18e3386951e2e75378b28da6d02ec529d"
EXPECTED_IMAGE_BYTES = 4260352
import hashlib, io, json, os, platform, time, uuid, zipfile
from contextlib import nullcontext
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps
import torch
from google.colab import files

PRAGMA_HEADLESS = os.environ.get("PRAGMA_HEADLESS") == "1"   # 1 = Colab CLI, sin navegador

def select_precision(cuda_available, capability=None):
    if not cuda_available:
        return {"device": "cpu", "dtype": "float32", "autocast": False}
    native_bf16 = int(capability[0]) >= 8
    return {"device": "cuda", "dtype": "bfloat16" if native_bf16 else "float16", "autocast": True}

CUDA_AVAILABLE = torch.cuda.is_available()
CUDA_CAPABILITY = torch.cuda.get_device_capability(0) if CUDA_AVAILABLE else None
PRECISION = select_precision(CUDA_AVAILABLE, CUDA_CAPABILITY)
DEVICE = PRECISION["device"]
TORCH_DTYPE = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}[PRECISION["dtype"]]

def inference_precision():
    return torch.autocast(device_type="cuda", dtype=TORCH_DTYPE) if DEVICE == "cuda" else nullcontext()

def synchronize():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
    DEVICE_NAME = torch.cuda.get_device_name(0)
else:
    DEVICE_NAME = "CPU"
    print("ADVERTENCIA: sin GPU. La corrida se registrará como REAL_CPU y no será comparable.")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = WORK_DIR / "runs_v13" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

def sha256_file(path, chunk=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()

def find_acceptance_image(folders):
    for folder in map(Path, folders):
        if not folder.is_dir():
            continue
        for path in sorted(folder.iterdir()):
            if (path.is_file() and path.suffix.lower() in (".jpg", ".jpeg")
                    and path.stat().st_size == EXPECTED_IMAGE_BYTES and sha256_file(path) == EXPECTED_IMAGE_SHA256):
                return path
    return None

IMAGE_PATH = find_acceptance_image([Path("/content"), Path("/mnt/data"), WORK_DIR])
if IMAGE_PATH is not None:
    print(f"Foto encontrada por su huella SHA-256: {IMAGE_PATH}")
elif PRAGMA_HEADLESS:
    raise FileNotFoundError("Modo sin navegador: sube antes la foto a /content.")
else:
    print("Selecciona la foto P1070614.JPG. Se validará por SHA-256; el nombre puede variar.")
    uploaded = files.upload()
    exact = [(n, p) for n, p in uploaded.items() if hashlib.sha256(p).hexdigest() == EXPECTED_IMAGE_SHA256]
    if len(exact) != 1:
        raise ValueError("No se recibió exactamente la fotografía de aceptación.")
    IMAGE_PATH = RUN_DIR / "P1070614.JPG"
    IMAGE_PATH.write_bytes(exact[0][1])

IMAGE_SHA256 = sha256_file(IMAGE_PATH)
assert IMAGE_SHA256 == EXPECTED_IMAGE_SHA256, "La foto no coincide con la aceptación acordada."
image = np.asarray(ImageOps.exif_transpose(Image.open(IMAGE_PATH)).convert("RGB"))
assert image.shape[:2] == (2248, 4000), image.shape
ENVIRONMENT = {
    "python": platform.python_version(), "torch": torch.__version__, "device": DEVICE, "device_name": DEVICE_NAME,
    "cuda_capability": CUDA_CAPABILITY, "dtype": PRECISION["dtype"], "sam2_commit": SAM2_COMMIT,
    "checkpoint_bytes": CHECKPOINT.stat().st_size, "checkpoint_sha256": sha256_file(CHECKPOINT),
    "image_sha256": IMAGE_SHA256, "image_size": [image.shape[1], image.shape[0]], "run_id": RUN_ID,
}
print(json.dumps(ENVIRONMENT, indent=2, ensure_ascii=False))

In [ ]:
# Celda 3 · prerregistro embebido (no editar) y compuerta de luma de TODOS los prompts
PREREG_FILE_SHA256 = "9953ed9cb22e1b54a2a7116450158f8d68d48269ec679b3e53309d839ff29fb8"
PREREG_TEXT = "{\n  \"schema\": \"pragma.aem1_preregistration\",\n  \"schema_version\": \"0.1.0\",\n  \"experiment\": \"A-E(−1) v1.3 · caso chica\",\n  \"status\": \"PREREGISTERED\",\n  \"frozen_on\": \"2026-09-25\",\n  \"responds_to\": [\n    \"dialogo/002_chatgpt_a_claude.md\",\n    \"dialogo/003_chatgpt_a_claude.md\"\n  ],\n  \"cross_audit\": {\n    \"by\": \"ChatGPT\",\n    \"letter\": \"dialogo/003_chatgpt_a_claude.md\",\n    \"letter_sha256\": \"dcc637a0fbdbad6aec4f43b62a1e4bf645605a13bc5ab56ec37cfd9fb166484e\",\n    \"inspected_package_sha256\": \"0453d7d271f260e5db4faef7b9a769dfd63386d6afdabca45d290aeb0737e8ff\",\n    \"inspected_prereg_content_sha256\": \"20d1f9f534ab941e0a278d6c218149c910c4abedfedb0fe25e17d3ba02c2bf1d\",\n    \"verdicts\": {\n      \"ZIP_INTEGRITY\": \"PASS\",\n      \"DESIGN_PROMPTS_SECOND_KEY\": \"PASS\",\n      \"H1/S1/P+1_PERTURBATIONS\": \"PASS\",\n      \"DECISION_A\": \"ACCEPT\",\n      \"DECISION_B\": \"ACCEPT_WITH_NAMING_CLARIFICATION\",\n      \"DECISION_C\": \"ACCEPT_WITH_CONTINUOUS_DIAGNOSTICS\",\n      \"DECISION_D\": \"ACCEPT_AS_DIAGNOSTIC\",\n      \"DECISION_E\": \"CHANGE_REQUIRED\",\n      \"BLIND_PROTOCOL_V2\": \"ACCEPT_AFTER_TECHNICAL_ADJUDICATION_CHANGE\"\n    },\n    \"changes_applied_before_any_run\": [\n      \"(e) BASE bit a bit → BASE_V2_REFERENCE (referencia normalizada y adjudicada), no doble llave heredada\",\n      \"protocolo v2 rev. 1 §5.2: discrepancia → adjudicación técnica; la persona usuaria conserva el veto\",\n      \"(b) perturbaciones con radio L∞ y distancia euclídea explícitos\",\n      \"(c) cobertura continua de O* y distancia al umbral en cada CONFLICT\",\n      \"(d) razones de propiedad |T∩R|/|T| y |T∩R|/|R| además de la de min()\",\n      \"descripción de P+1: botón/overol en el torso de la chica (sin cambiar coordenadas)\",\n      \"hipótesis H-G1…H-G4 de ChatGPT registradas\"\n    ],\n    \"result\": \"AEM1_v1.3 = GO_TO_BUILD tras aplicar los cambios (ChatGPT 003)\"\n  },\n  \"generated_by\": \"work/design_aem1_v1_3.py\",\n  \"image\": {\n    \"sha256\": \"8f6e3b6f5013265a45c7e89121e3f0a18e3386951e2e75378b28da6d02ec529d\",\n    \"size\": [\n      4000,\n      2248\n    ],\n    \"orientation\": \"EXIF aplicado; sin redimensionar\"\n  },\n  \"sam2_freeze\": {\n    \"SAM2_GIT_COMMIT\": \"2b90b9f5ceec907a1c18123530e92e794ad901a4\",\n    \"MODEL_CONFIG\": \"configs/sam2.1/sam2.1_hiera_l.yaml\",\n    \"CHECKPOINT\": \"sam2.1_hiera_large.pt\",\n    \"CHECKPOINT_SHA256\": \"2647878d5dfa5098f2f8649825738a9345572bae2d4350a2468587ece47dd318\",\n    \"CHECKPOINT_BYTES\": 898083611,\n    \"dtype\": \"bfloat16 (autocast CUDA), como la corrida 1\",\n    \"gate\": \"el cuaderno bloquea antes de generar si el commit, el checkpoint o la foto difieren\"\n  },\n  \"base_config\": {\n    \"source_notebook\": \"outputs/PRAGMA_A-E-menos-1_diagnostico_caso_chica_v1_2.ipynb\",\n    \"source_notebook_sha256\": \"06315e0fe15b84b446570903d7a6c5c219df5ea2ecf4577429cf107cdcdef0da\",\n    \"run1_config_digest\": \"fd29b18bbe50cf8236f41057567265e6083d13465511e4882cd8e533af58aa71\",\n    \"BOX\": [\n      2100,\n      300,\n      3500,\n      2247\n    ],\n    \"prompts\": [\n      {\n        \"id\": \"P+1\",\n        \"xy\": [\n          2588,\n          1785\n        ],\n        \"description\": \"botón/overol en el torso de la chica\",\n        \"patch_luma_mean\": 249.3,\n        \"patch_luma_std\": 6.2,\n        \"description_v1_2\": \"torso de la chica\"\n      },\n      {\n        \"id\": \"P-1\",\n        \"xy\": [\n          2350,\n          900\n        ],\n        \"description\": \"torso floral posterior\",\n        \"patch_luma_mean\": 125.6,\n        \"patch_luma_std\": 71.0\n      },\n      {\n        \"id\": \"P-2\",\n        \"xy\": [\n          2640,\n          430\n        ],\n        \"description\": \"cabello recogido posterior (v1.1: antes borde con pared)\",\n        \"patch_luma_mean\": 56.8,\n        \"patch_luma_std\": 44.9\n      },\n      {\n        \"id\": \"P-3\",\n        \"xy\": [\n          2400,\n          810\n        ],\n        \"description\": \"hombro posterior, tela oscura (v1.1: antes pared)\",\n        \"patch_luma_mean\": 90.9,\n        \"patch_luma_std\": 53.0\n      }\n    ],\n    \"holdouts\": {\n      \"keep_subject\": [\n        {\n          \"id\": \"K1\",\n          \"xy\": [\n            2835,\n            635\n          ],\n          \"description\": \"cara\",\n          \"patch_luma_mean\": 245.8,\n          \"patch_luma_std\": 2.9\n        },\n        {\n          \"id\": \"K2\",\n          \"xy\": [\n            2730,\n            650\n          ],\n          \"description\": \"cabello frontal\",\n          \"patch_luma_mean\": 135.3,\n          \"patch_luma_std\": 51.7\n        },\n        {\n          \"id\": \"K3\",\n          \"xy\": [\n            2435,\n            1135\n          ],\n          \"description\": \"hombro/ropa izquierda\",\n          \"patch_luma_mean\": 101.4,\n          \"patch_luma_std\": 57.1\n        },\n        {\n          \"id\": \"K4\",\n          \"xy\": [\n            3185,\n            985\n          ],\n          \"description\": \"mano levantada\",\n          \"patch_luma_mean\": 254.3,\n          \"patch_luma_std\": 0.8\n        },\n        {\n          \"id\": \"K5\",\n          \"xy\": [\n            3285,\n            1335\n          ],\n          \"description\": \"manga/brazo derecho\",\n          \"patch_luma_mean\": 101.7,\n          \"patch_luma_std\": 53.1\n        },\n        {\n          \"id\": \"K6\",\n          \"xy\": [\n            2235,\n            1935\n          ],\n          \"description\": \"mano que cuelga\",\n          \"patch_luma_mean\": 220.3,\n          \"patch_luma_std\": 23.7\n        },\n        {\n          \"id\": \"K7\",\n          \"xy\": [\n            2785,\n            2035\n          ],\n          \"description\": \"torso inferior\",\n          \"patch_luma_mean\": 252.9,\n          \"patch_luma_std\": 1.7\n        }\n      ],\n      \"drop_other_person\": [\n        {\n          \"id\": \"O1\",\n          \"xy\": [\n            2460,\n            860\n          ],\n          \"description\": \"ropa floral posterior\",\n          \"patch_luma_mean\": 179.8,\n          \"patch_luma_std\": 57.1\n        },\n        {\n          \"id\": \"O2\",\n          \"xy\": [\n            2700,\n            380\n          ],\n          \"description\": \"cabello recogido posterior (v1.1: antes pared)\",\n          \"patch_luma_mean\": 78.7,\n          \"patch_luma_std\": 52.6\n        },\n        {\n          \"id\": \"O3\",\n          \"xy\": [\n            2735,\n            360\n          ],\n          \"description\": \"cabello recogido posterior, arriba (v1.2: antes zona de contacto)\",\n          \"patch_luma_mean\": 57.1,\n          \"patch_luma_std\": 43.5\n        },\n        {\n          \"id\": \"O4\",\n          \"xy\": [\n            2325,\n            965\n          ],\n          \"description\": \"blusa floral posterior, interior (v1.2: antes borde)\",\n          \"patch_luma_mean\": 185.1,\n          \"patch_luma_std\": 36.5\n        }\n      ],\n      \"drop_background\": [\n        {\n          \"id\": \"B1\",\n          \"xy\": [\n            2185,\n            435\n          ],\n          \"description\": \"cuadro\",\n          \"patch_luma_mean\": 203.4,\n          \"patch_luma_std\": 28.8\n        },\n        {\n          \"id\": \"B2\",\n          \"xy\": [\n            1735,\n            1635\n          ],\n          \"description\": \"mesa/mantel\",\n          \"patch_luma_mean\": 182.9,\n          \"patch_luma_std\": 35.6\n        },\n        {\n          \"id\": \"B3\",\n          \"xy\": [\n            3485,\n            1485\n          ],\n          \"description\": \"sofá/fondo\",\n          \"patch_luma_mean\": 49.6,\n          \"patch_luma_std\": 39.4\n        }\n      ]\n    },\n    \"patch_radius\": 6,\n    \"keep_min_coverage\": 0.8,\n    \"drop_max_coverage\": 0.2\n  },\n  \"new_prompts\": {\n    \"H1\": {\n      \"xy\": [\n        3072,\n        592\n      ],\n      \"owner_expected\": \"chica\",\n      \"material\": \"pelo oscuro\",\n      \"declared_window_xyxy\": [\n        2940,\n        400,\n        3200,\n        900\n      ],\n      \"window_rationale\": \"pelo de la chica a la derecha de su cabeza (lado opuesto al moño), fuera de la caja de contacto\",\n      \"selection_rule\": \"pragma_ae.aem1_v13.select_safe_point (rejilla de 2 px)\",\n      \"safe_square_half_px\": 30,\n      \"safe_square_dark_fraction\": 0.9817,\n      \"contact_box_margin_linf_px\": 173,\n      \"frontier_distance_lower_bound_px\": 173,\n      \"nearest_holdout\": {\n        \"id\": \"K1\",\n        \"distance_px\": 240.9\n      },\n      \"nearest_prompt\": {\n        \"id\": \"P-2\",\n        \"distance_px\": 461.4\n      },\n      \"patch_luma_mean\": 70.8,\n      \"patch_luma_std\": 48.4,\n      \"image_sha256\": \"8f6e3b6f5013265a45c7e89121e3f0a18e3386951e2e75378b28da6d02ec529d\",\n      \"owner_verified_by\": \"auditor IA Claude en la lámina privada de diseño\",\n      \"owner_second_key\": \"PASS (ChatGPT 003: el punto y sus 8 perturbaciones sobre la chica)\"\n    },\n    \"S1\": {\n      \"xy\": [\n        2230,\n        1686\n      ],\n      \"owner_expected\": \"chica\",\n      \"material\": \"manga oscura\",\n      \"declared_window_xyxy\": [\n        2120,\n        1350,\n        2340,\n        1850\n      ],\n      \"window_rationale\": \"manga oscura del brazo que cuelga, por debajo de la caja de contacto\",\n      \"selection_rule\": \"pragma_ae.aem1_v13.select_safe_point (rejilla de 2 px)\",\n      \"safe_square_half_px\": 85,\n      \"safe_square_dark_fraction\": 0.9842,\n      \"contact_box_margin_linf_px\": 387,\n      \"frontier_distance_lower_bound_px\": 387,\n      \"nearest_holdout\": {\n        \"id\": \"K6\",\n        \"distance_px\": 249.1\n      },\n      \"nearest_prompt\": {\n        \"id\": \"P+1\",\n        \"distance_px\": 371.4\n      },\n      \"patch_luma_mean\": 66.9,\n      \"patch_luma_std\": 49.4,\n      \"image_sha256\": \"8f6e3b6f5013265a45c7e89121e3f0a18e3386951e2e75378b28da6d02ec529d\",\n      \"owner_verified_by\": \"auditor IA Claude en la lámina privada de diseño\",\n      \"owner_second_key\": \"PASS (ChatGPT 003: el punto y sus 8 perturbaciones sobre la chica)\"\n    }\n  },\n  \"safety_constants\": {\n    \"contact_box_xyxy\": [\n      2150,\n      250,\n      2900,\n      1300\n    ],\n    \"contact_box_claim\": \"toda la frontera visible chica↔persona posterior está dentro de esta caja; su margen es cota inferior de la distancia a la frontera\",\n    \"contact_margin_min_linf_px\": 40,\n    \"perturbed_contact_margin_min_linf_px\": 25,\n    \"holdout_distance_min_px\": 100,\n    \"perturbed_holdout_distance_min_px\": 78,\n    \"prompt_distance_min_px\": 100,\n    \"dark_luma_threshold\": 110,\n    \"dark_blur_radius\": 4,\n    \"safe_square_dark_fraction_min\": 0.98,\n    \"safe_square_half_min_px\": 21,\n    \"perturbed_patch_dark_fraction_min\": 0.9,\n    \"luma\": \"media de los canales RGB en el parche 13×13, como el preflight v1.2\",\n    \"runtime_binding\": \"el cuaderno recalcula la luma de TODO prompt (base y perturbado) y bloquea si difiere > 3,0 del valor registrado aquí\"\n  },\n  \"invalid_perturbation\": \"un caso INVALID_PERTURBATION no se ejecuta y nunca cuenta como evidencia contra SAM 2\",\n  \"branches\": {\n    \"BASE\": {\n      \"purpose\": \"replicar la corrida 1 (los mismos prompts, la misma caja y el mismo congelado)\",\n      \"protocols\": {\n        \"point\": {\n          \"points\": [\n            \"P+1\"\n          ],\n          \"labels\": [\n            1\n          ],\n          \"box\": null,\n          \"multimask_output\": true,\n          \"outputs\": 3\n        },\n        \"box\": {\n          \"points\": [],\n          \"labels\": [],\n          \"box\": \"BOX\",\n          \"multimask_output\": true,\n          \"outputs\": 3\n        },\n        \"point+corrections:s{0,1,2}\": {\n          \"points\": [\n            \"P+1\",\n            \"P-1\",\n            \"P-2\",\n            \"P-3\"\n          ],\n          \"labels\": [\n            1,\n            0,\n            0,\n            0\n          ],\n          \"box\": null,\n          \"seed\": \"point\",\n          \"note\": \"mask_input = low_res_logits[s] de la semilla BASE del mismo tipo; multimask_output=False\"\n        },\n        \"box+corrections:s{0,1,2}\": {\n          \"points\": [\n            \"P+1\",\n            \"P-1\",\n            \"P-2\",\n            \"P-3\"\n          ],\n          \"labels\": [\n            1,\n            0,\n            0,\n            0\n          ],\n          \"box\": \"BOX\",\n          \"seed\": \"box\",\n          \"note\": \"mask_input = low_res_logits[s] de la semilla BASE del mismo tipo; multimask_output=False\"\n        }\n      },\n      \"reproduction_check\": {\n        \"rule\": \"packed_mask_sha256 de cada candidata BASE frente a la corrida 1\",\n        \"labels\": {\n          \"BIT_EXACT\": \"máscara idéntica: su referencia es BASE_V2_REFERENCE (adjudicación de la corrida 1 normalizada a las definiciones v2), NO una doble llave v2 heredada; no vuelve al paquete ciego\",\n          \"NOT_BIT_EXACT\": \"las que difieran entran al paquete ciego con etiquetas nuevas\"\n        },\n        \"reference\": {\n          \"file\": \"auditoria/aem1_20260925T062504Z_256dba9f/BASE_V2_REFERENCE.json\",\n          \"sha256\": \"c1f01dd9d912b721be6470e5f9be44bf2a2b48efcaab7bf271fc884e37800b66\",\n          \"kind\": \"REFERENCIA_NORMALIZADA_ADJUDICADA\",\n          \"why\": \"ChatGPT 003 (e): BIT_EXACT_MASK ≠ BIT_EXACT_JUDGMENT_UNDER_NEW_PROTOCOL\"\n        },\n        \"run1_packed_mask_sha256\": {\n          \"box#0\": \"2e3cc1c8ef21fa99919506b1e2fbb4d8e24e38ac93fce7e31e272fb2e05590aa\",\n          \"box#1\": \"9aa2c41783b0593ca8d702104b1d8756b9c0c08450c1bf9b0e9afdff2ac624e8\",\n          \"box#2\": \"7fa08b3ddb75b6e585811caea4d65ffab19f38a621af64cdf2ade28cce011466\",\n          \"box+corrections:s0#0\": \"29ad728d537c3a62c717a468f365a5582a3bf09e080fc40000ca1c825e5a620b\",\n          \"box+corrections:s1#0\": \"8ae7900637c5cde7d03b215ff3d29bf34f780b4a5a65c284877c6aecea11592d\",\n          \"box+corrections:s2#0\": \"7dc75cb30c37f1ce3fc792ac9f0b4c5ca894fab1b7a364b6065938d4f42084df\",\n          \"point#0\": \"2cb424a207262e9ceeababd3c5f23afa3509562368f3f8782a58e2ecad889972\",\n          \"point#1\": \"c3c44fd38281b0ee5aaa085b630df97ed05b51f2292ddaf96ddcb98562af07bc\",\n          \"point#2\": \"eed5e58e37125863f8412d69ba685c66b4efb38d05e567d924ed6f54bf66c33c\",\n          \"point+corrections:s0#0\": \"020cb1d3e54c60d619c52427a29f7a645add824b964e16d6c4298cc9803ba03a\",\n          \"point+corrections:s1#0\": \"c5ccae5a5ec06d1e4fad8348987203d91390a6ae97494d7ad669cc0392151006\",\n          \"point+corrections:s2#0\": \"4c58cfd9b2f878af4ff4f3cff9e804377dc460c0bb340b9b76d03e216432d3fc\"\n        }\n      }\n    },\n    \"+POS_HAIR\": {\n      \"purpose\": \"efecto causal de añadir H1 como positivo(s) a las correcciones\",\n      \"only_change_vs_BASE\": \"H1 con etiqueta 1 en la llamada de corrección; semillas idénticas a BASE\",\n      \"protocols\": {\n        \"point+corrections:s{0,1,2}\": {\n          \"points\": [\n            \"P+1\",\n            \"H1\",\n            \"P-1\",\n            \"P-2\",\n            \"P-3\"\n          ],\n          \"labels\": [\n            1,\n            1,\n            0,\n            0,\n            0\n          ],\n          \"box\": null,\n          \"seed\": \"point (BASE)\",\n          \"note\": \"mask_input = low_res_logits[s] de la semilla BASE del mismo tipo; multimask_output=False\"\n        },\n        \"box+corrections:s{0,1,2}\": {\n          \"points\": [\n            \"P+1\",\n            \"H1\",\n            \"P-1\",\n            \"P-2\",\n            \"P-3\"\n          ],\n          \"labels\": [\n            1,\n            1,\n            0,\n            0,\n            0\n          ],\n          \"box\": \"BOX\",\n          \"seed\": \"box (BASE)\",\n          \"note\": \"mask_input = low_res_logits[s] de la semilla BASE del mismo tipo; multimask_output=False\"\n        }\n      },\n      \"why_not_point_or_box_alone\": \"los positivos nuevos contrarrestan a los negativos; sin negativos, box ya incluye todo lo oscuro (corrida 1) y point no lleva negativos\"\n    },\n    \"+POS_SLEEVE\": {\n      \"purpose\": \"efecto causal de añadir S1 como positivo(s) a las correcciones\",\n      \"only_change_vs_BASE\": \"S1 con etiqueta 1 en la llamada de corrección; semillas idénticas a BASE\",\n      \"protocols\": {\n        \"point+corrections:s{0,1,2}\": {\n          \"points\": [\n            \"P+1\",\n            \"S1\",\n            \"P-1\",\n            \"P-2\",\n            \"P-3\"\n          ],\n          \"labels\": [\n            1,\n            1,\n            0,\n            0,\n            0\n          ],\n          \"box\": null,\n          \"seed\": \"point (BASE)\",\n          \"note\": \"mask_input = low_res_logits[s] de la semilla BASE del mismo tipo; multimask_output=False\"\n        },\n        \"box+corrections:s{0,1,2}\": {\n          \"points\": [\n            \"P+1\",\n            \"S1\",\n            \"P-1\",\n            \"P-2\",\n            \"P-3\"\n          ],\n          \"labels\": [\n            1,\n            1,\n            0,\n            0,\n            0\n          ],\n          \"box\": \"BOX\",\n          \"seed\": \"box (BASE)\",\n          \"note\": \"mask_input = low_res_logits[s] de la semilla BASE del mismo tipo; multimask_output=False\"\n        }\n      },\n      \"why_not_point_or_box_alone\": \"los positivos nuevos contrarrestan a los negativos; sin negativos, box ya incluye todo lo oscuro (corrida 1) y point no lleva negativos\"\n    },\n    \"+POS_HAIR+SLEEVE\": {\n      \"purpose\": \"efecto causal de añadir H1 y S1 como positivo(s) a las correcciones\",\n      \"only_change_vs_BASE\": \"H1 y S1 con etiqueta 1 en la llamada de corrección; semillas idénticas a BASE\",\n      \"protocols\": {\n        \"point+corrections:s{0,1,2}\": {\n          \"points\": [\n            \"P+1\",\n            \"H1\",\n            \"S1\",\n            \"P-1\",\n            \"P-2\",\n            \"P-3\"\n          ],\n          \"labels\": [\n            1,\n            1,\n            1,\n            0,\n            0,\n            0\n          ],\n          \"box\": null,\n          \"seed\": \"point (BASE)\",\n          \"note\": \"mask_input = low_res_logits[s] de la semilla BASE del mismo tipo; multimask_output=False\"\n        },\n        \"box+corrections:s{0,1,2}\": {\n          \"points\": [\n            \"P+1\",\n            \"H1\",\n            \"S1\",\n            \"P-1\",\n            \"P-2\",\n            \"P-3\"\n          ],\n          \"labels\": [\n            1,\n            1,\n            1,\n            0,\n            0,\n            0\n          ],\n          \"box\": \"BOX\",\n          \"seed\": \"box (BASE)\",\n          \"note\": \"mask_input = low_res_logits[s] de la semilla BASE del mismo tipo; multimask_output=False\"\n        }\n      },\n      \"why_not_point_or_box_alone\": \"los positivos nuevos contrarrestan a los negativos; sin negativos, box ya incluye todo lo oscuro (corrida 1) y point no lleva negativos\"\n    },\n    \"RECIPROCAL_POSTERIOR\": {\n      \"purpose\": \"¿puede SAM 2 segmentar de forma estable a la persona posterior cuando se le pide directamente?\",\n      \"mirror_of\": \"point y point+corrections, con los papeles invertidos\",\n      \"protocols\": {\n        \"R-point\": {\n          \"points\": [\n            \"P-1\"\n          ],\n          \"labels\": [\n            1\n          ],\n          \"box\": null,\n          \"multimask_output\": true,\n          \"outputs\": 3\n        },\n        \"R-corrections:s{0,1,2}\": {\n          \"points\": [\n            \"P-1\",\n            \"P-2\",\n            \"P-3\",\n            \"P+1\",\n            \"H1\",\n            \"S1\"\n          ],\n          \"labels\": [\n            1,\n            1,\n            1,\n            0,\n            0,\n            0\n          ],\n          \"box\": null,\n          \"seed\": \"R-point\",\n          \"note\": \"mask_input = low_res_logits[s] de R-point; multimask_output=False\"\n        }\n      },\n      \"sentinels_swapped\": {\n        \"keep\": \"O1–O4\",\n        \"drop_other_person\": \"K1–K7\",\n        \"drop_background\": \"B1–B3\"\n      },\n      \"reference_mask\": \"R_ref = la semilla R-corrections con más sentinelas O cubiertos (≥ 0,80); empate → menor índice\",\n      \"never\": \"no se usa para reparar la máscara de la chica (nada de objetivo − posterior)\"\n    },\n    \"PERTURB_POINT\": {\n      \"offsets_px\": [\n        [\n          15,\n          0\n        ],\n        [\n          -15,\n          0\n        ],\n        [\n          0,\n          15\n        ],\n        [\n          0,\n          -15\n        ],\n        [\n          15,\n          15\n        ],\n        [\n          15,\n          -15\n        ],\n        [\n          -15,\n          15\n        ],\n        [\n          -15,\n          -15\n        ]\n      ],\n      \"neighbourhood\": \"anillo L∞ de radio 15 px, determinista: 4 axiales (euclídea 15 px) y 4 diagonales (euclídea 15·√2 ≈ 21,21 px); no son desplazamientos equivalentes\",\n      \"linf_radius_px\": 15,\n      \"euclidean_px\": {\n        \"axial\": 15.0,\n        \"diagonal\": 21.21\n      },\n      \"one_at_a_time\": true,\n      \"targets\": {\n        \"P+1\": {\n          \"base\": \"P+1\",\n          \"base_xy\": [\n            2588,\n            1785\n          ],\n          \"perturbations\": [\n            {\n              \"id\": \"d+15+0\",\n              \"dx\": 15,\n              \"dy\": 0,\n              \"xy\": [\n                2603,\n                1785\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 15.0,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 252.1,\n              \"patch_luma_std\": 2.3\n            },\n            {\n              \"id\": \"d-15+0\",\n              \"dx\": -15,\n              \"dy\": 0,\n              \"xy\": [\n                2573,\n                1785\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 15.0,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 248.6,\n              \"patch_luma_std\": 6.2\n            },\n            {\n              \"id\": \"d+0+15\",\n              \"dx\": 0,\n              \"dy\": 15,\n              \"xy\": [\n                2588,\n                1800\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 15.0,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 230.9,\n              \"patch_luma_std\": 13.2\n            },\n            {\n              \"id\": \"d+0-15\",\n              \"dx\": 0,\n              \"dy\": -15,\n              \"xy\": [\n                2588,\n                1770\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 15.0,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 251.9,\n              \"patch_luma_std\": 2.0\n            },\n            {\n              \"id\": \"d+15+15\",\n              \"dx\": 15,\n              \"dy\": 15,\n              \"xy\": [\n                2603,\n                1800\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 21.21,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 245.7,\n              \"patch_luma_std\": 11.5\n            },\n            {\n              \"id\": \"d+15-15\",\n              \"dx\": 15,\n              \"dy\": -15,\n              \"xy\": [\n                2603,\n                1770\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 21.21,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 252.4,\n              \"patch_luma_std\": 2.1\n            },\n            {\n              \"id\": \"d-15+15\",\n              \"dx\": -15,\n              \"dy\": 15,\n              \"xy\": [\n                2573,\n                1800\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 21.21,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 219.6,\n              \"patch_luma_std\": 22.6\n            },\n            {\n              \"id\": \"d-15-15\",\n              \"dx\": -15,\n              \"dy\": -15,\n              \"xy\": [\n                2573,\n                1770\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 21.21,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 251.7,\n              \"patch_luma_std\": 2.2\n            }\n          ],\n          \"valid_count\": 8,\n          \"branch\": \"BASE\",\n          \"protocols\": [\n            \"point\",\n            \"point+corrections\"\n          ],\n          \"chain\": \"el punto perturbado sustituye a P+1 en la semilla (point, multimask) y en la corrección\"\n        },\n        \"H1\": {\n          \"base\": \"H1\",\n          \"base_xy\": [\n            3072,\n            592\n          ],\n          \"perturbations\": [\n            {\n              \"id\": \"d+15+0\",\n              \"dx\": 15,\n              \"dy\": 0,\n              \"xy\": [\n                3087,\n                592\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 15.0,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 51.4,\n              \"patch_luma_std\": 41.8\n            },\n            {\n              \"id\": \"d-15+0\",\n              \"dx\": -15,\n              \"dy\": 0,\n              \"xy\": [\n                3057,\n                592\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 15.0,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 73.9,\n              \"patch_luma_std\": 53.6\n            },\n            {\n              \"id\": \"d+0+15\",\n              \"dx\": 0,\n              \"dy\": 15,\n              \"xy\": [\n                3072,\n                607\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 15.0,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 87.3,\n              \"patch_luma_std\": 54.1\n            },\n            {\n              \"id\": \"d+0-15\",\n              \"dx\": 0,\n              \"dy\": -15,\n              \"xy\": [\n                3072,\n                577\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 15.0,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 78.2,\n              \"patch_luma_std\": 54.3\n            },\n            {\n              \"id\": \"d+15+15\",\n              \"dx\": 15,\n              \"dy\": 15,\n              \"xy\": [\n                3087,\n                607\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 21.21,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 64.8,\n              \"patch_luma_std\": 42.1\n            },\n            {\n              \"id\": \"d+15-15\",\n              \"dx\": 15,\n              \"dy\": -15,\n              \"xy\": [\n                3087,\n                577\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 21.21,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 41.2,\n              \"patch_luma_std\": 34.9\n            },\n            {\n              \"id\": \"d-15+15\",\n              \"dx\": -15,\n              \"dy\": 15,\n              \"xy\": [\n                3057,\n                607\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 21.21,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 91.5,\n              \"patch_luma_std\": 60.5\n            },\n            {\n              \"id\": \"d-15-15\",\n              \"dx\": -15,\n              \"dy\": -15,\n              \"xy\": [\n                3057,\n                577\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 21.21,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 64.1,\n              \"patch_luma_std\": 51.9\n            }\n          ],\n          \"valid_count\": 8,\n          \"branch\": \"+POS_HAIR+SLEEVE\",\n          \"protocols\": [\n            \"point+corrections\",\n            \"box+corrections\"\n          ],\n          \"chain\": \"semillas de BASE sin cambios; solo H1 se perturba (S1 fijo)\"\n        },\n        \"S1\": {\n          \"base\": \"S1\",\n          \"base_xy\": [\n            2230,\n            1686\n          ],\n          \"perturbations\": [\n            {\n              \"id\": \"d+15+0\",\n              \"dx\": 15,\n              \"dy\": 0,\n              \"xy\": [\n                2245,\n                1686\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 15.0,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 78.9,\n              \"patch_luma_std\": 51.7\n            },\n            {\n              \"id\": \"d-15+0\",\n              \"dx\": -15,\n              \"dy\": 0,\n              \"xy\": [\n                2215,\n                1686\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 15.0,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 74.0,\n              \"patch_luma_std\": 51.5\n            },\n            {\n              \"id\": \"d+0+15\",\n              \"dx\": 0,\n              \"dy\": 15,\n              \"xy\": [\n                2230,\n                1701\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 15.0,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 76.1,\n              \"patch_luma_std\": 53.2\n            },\n            {\n              \"id\": \"d+0-15\",\n              \"dx\": 0,\n              \"dy\": -15,\n              \"xy\": [\n                2230,\n                1671\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 15.0,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 68.7,\n              \"patch_luma_std\": 49.2\n            },\n            {\n              \"id\": \"d+15+15\",\n              \"dx\": 15,\n              \"dy\": 15,\n              \"xy\": [\n                2245,\n                1701\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 21.21,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 73.0,\n              \"patch_luma_std\": 47.8\n            },\n            {\n              \"id\": \"d+15-15\",\n              \"dx\": 15,\n              \"dy\": -15,\n              \"xy\": [\n                2245,\n                1671\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 21.21,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 70.7,\n              \"patch_luma_std\": 50.3\n            },\n            {\n              \"id\": \"d-15+15\",\n              \"dx\": -15,\n              \"dy\": 15,\n              \"xy\": [\n                2215,\n                1701\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 21.21,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 69.0,\n              \"patch_luma_std\": 49.4\n            },\n            {\n              \"id\": \"d-15-15\",\n              \"dx\": -15,\n              \"dy\": -15,\n              \"xy\": [\n                2215,\n                1671\n              ],\n              \"linf_px\": 15,\n              \"euclidean_px\": 21.21,\n              \"valid\": true,\n              \"reasons\": [],\n              \"patch_luma_mean\": 67.2,\n              \"patch_luma_std\": 44.4\n            }\n          ],\n          \"valid_count\": 8,\n          \"branch\": \"+POS_HAIR+SLEEVE\",\n          \"protocols\": [\n            \"point+corrections\",\n            \"box+corrections\"\n          ],\n          \"chain\": \"semillas de BASE sin cambios; solo S1 se perturba (H1 fijo)\"\n        }\n      }\n    },\n    \"PERTURB_BOX\": {\n      \"base_box_xyxy\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"protocols\": [\n        \"box\",\n        \"box+corrections\"\n      ],\n      \"chain\": \"la caja perturbada sustituye a la base en la semilla (box, multimask) y en la corrección\",\n      \"coordinates\": \"inclusivas: 0 ≤ x ≤ 3999, 0 ≤ y ≤ 2247 (y2 = 2247 es la última fila)\",\n      \"perturbations\": [\n        {\n          \"id\": \"T+15+0\",\n          \"family\": \"TRANSLATE\",\n          \"box\": [\n            2115,\n            300,\n            3515,\n            2247\n          ],\n          \"valid\": true,\n          \"reason\": null\n        },\n        {\n          \"id\": \"T-15+0\",\n          \"family\": \"TRANSLATE\",\n          \"box\": [\n            2085,\n            300,\n            3485,\n            2247\n          ],\n          \"valid\": true,\n          \"reason\": null\n        },\n        {\n          \"id\": \"T+0-15\",\n          \"family\": \"TRANSLATE\",\n          \"box\": [\n            2100,\n            285,\n            3500,\n            2232\n          ],\n          \"valid\": true,\n          \"reason\": null\n        },\n        {\n          \"id\": \"T+0+15\",\n          \"family\": \"TRANSLATE\",\n          \"box\": [\n            2100,\n            315,\n            3500,\n            2262\n          ],\n          \"valid\": false,\n          \"reason\": \"INVALID_PERTURBATION: la caja trasladada sale de la imagen\"\n        },\n        {\n          \"id\": \"E15\",\n          \"family\": \"EXPAND\",\n          \"box\": [\n            2085,\n            285,\n            3515,\n            2247\n          ],\n          \"valid\": true,\n          \"reason\": null,\n          \"sides_fixed_at_image_border\": [\n            \"y2\"\n          ]\n        },\n        {\n          \"id\": \"C15\",\n          \"family\": \"CONTRACT\",\n          \"box\": [\n            2115,\n            315,\n            3485,\n            2232\n          ],\n          \"valid\": true,\n          \"reason\": null\n        }\n      ],\n      \"valid_count\": 5\n    }\n  },\n  \"combination\": \"NINGUNA en v1.3 (ChatGPT 002: solo si se prerregistra; no se prerregistra)\",\n  \"candidate_counts\": {\n    \"BASE\": 12,\n    \"+POS\": 18,\n    \"RECIPROCAL\": 6,\n    \"PERTURB_POINT\": 144,\n    \"PERTURB_BOX\": 30,\n    \"valid_point_perturbations\": 24,\n    \"total_masks\": 210,\n    \"calls\": 178\n  },\n  \"call_plan\": [\n    {\n      \"call_id\": \"BASE|point\",\n      \"branch\": \"BASE\",\n      \"protocol\": \"point\",\n      \"point_ids\": [\n        \"P+1\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ]\n      ],\n      \"labels\": [\n        1\n      ],\n      \"box\": null,\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"BASE|point|0\",\n        \"BASE|point|1\",\n        \"BASE|point|2\"\n      ]\n    },\n    {\n      \"call_id\": \"BASE|box\",\n      \"branch\": \"BASE\",\n      \"protocol\": \"box\",\n      \"point_ids\": [],\n      \"points\": [],\n      \"labels\": [],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"BASE|box|0\",\n        \"BASE|box|1\",\n        \"BASE|box|2\"\n      ]\n    },\n    {\n      \"call_id\": \"BASE|point+corrections|s0\",\n      \"branch\": \"BASE\",\n      \"protocol\": \"point+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"BASE|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"BASE|point+corrections|s1\",\n      \"branch\": \"BASE\",\n      \"protocol\": \"point+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"BASE|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"BASE|point+corrections|s2\",\n      \"branch\": \"BASE\",\n      \"protocol\": \"point+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"BASE|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"BASE|box+corrections|s0\",\n      \"branch\": \"BASE\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"BASE|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"BASE|box+corrections|s1\",\n      \"branch\": \"BASE\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"BASE|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"BASE|box+corrections|s2\",\n      \"branch\": \"BASE\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"BASE|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR|point+corrections|s0\",\n      \"branch\": \"+POS_HAIR\",\n      \"protocol\": \"point+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR|point+corrections|s1\",\n      \"branch\": \"+POS_HAIR\",\n      \"protocol\": \"point+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR|point+corrections|s2\",\n      \"branch\": \"+POS_HAIR\",\n      \"protocol\": \"point+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR|box+corrections|s0\",\n      \"branch\": \"+POS_HAIR\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR|box+corrections|s1\",\n      \"branch\": \"+POS_HAIR\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR|box+corrections|s2\",\n      \"branch\": \"+POS_HAIR\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_SLEEVE|point+corrections|s0\",\n      \"branch\": \"+POS_SLEEVE\",\n      \"protocol\": \"point+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_SLEEVE|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_SLEEVE|point+corrections|s1\",\n      \"branch\": \"+POS_SLEEVE\",\n      \"protocol\": \"point+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_SLEEVE|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_SLEEVE|point+corrections|s2\",\n      \"branch\": \"+POS_SLEEVE\",\n      \"protocol\": \"point+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_SLEEVE|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_SLEEVE|box+corrections|s0\",\n      \"branch\": \"+POS_SLEEVE\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_SLEEVE|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_SLEEVE|box+corrections|s1\",\n      \"branch\": \"+POS_SLEEVE\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_SLEEVE|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_SLEEVE|box+corrections|s2\",\n      \"branch\": \"+POS_SLEEVE\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_SLEEVE|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR+SLEEVE|point+corrections|s0\",\n      \"branch\": \"+POS_HAIR+SLEEVE\",\n      \"protocol\": \"point+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR+SLEEVE|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR+SLEEVE|point+corrections|s1\",\n      \"branch\": \"+POS_HAIR+SLEEVE\",\n      \"protocol\": \"point+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR+SLEEVE|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR+SLEEVE|point+corrections|s2\",\n      \"branch\": \"+POS_HAIR+SLEEVE\",\n      \"protocol\": \"point+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR+SLEEVE|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR+SLEEVE|box+corrections|s0\",\n      \"branch\": \"+POS_HAIR+SLEEVE\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR+SLEEVE|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR+SLEEVE|box+corrections|s1\",\n      \"branch\": \"+POS_HAIR+SLEEVE\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR+SLEEVE|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"+POS_HAIR+SLEEVE|box+corrections|s2\",\n      \"branch\": \"+POS_HAIR+SLEEVE\",\n      \"protocol\": \"box+corrections\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"+POS_HAIR+SLEEVE|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"RECIPROCAL|R-point\",\n      \"branch\": \"RECIPROCAL_POSTERIOR\",\n      \"protocol\": \"R-point\",\n      \"point_ids\": [\n        \"P-1\"\n      ],\n      \"points\": [\n        [\n          2350,\n          900\n        ]\n      ],\n      \"labels\": [\n        1\n      ],\n      \"box\": null,\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"RECIPROCAL|R-point|0\",\n        \"RECIPROCAL|R-point|1\",\n        \"RECIPROCAL|R-point|2\"\n      ]\n    },\n    {\n      \"call_id\": \"RECIPROCAL|R-corrections|s0\",\n      \"branch\": \"RECIPROCAL_POSTERIOR\",\n      \"protocol\": \"R-corrections\",\n      \"point_ids\": [\n        \"P-1\",\n        \"P-2\",\n        \"P-3\",\n        \"P+1\",\n        \"H1\",\n        \"S1\"\n      ],\n      \"points\": [\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ],\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"RECIPROCAL|R-point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"RECIPROCAL|R-corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"RECIPROCAL|R-corrections|s1\",\n      \"branch\": \"RECIPROCAL_POSTERIOR\",\n      \"protocol\": \"R-corrections\",\n      \"point_ids\": [\n        \"P-1\",\n        \"P-2\",\n        \"P-3\",\n        \"P+1\",\n        \"H1\",\n        \"S1\"\n      ],\n      \"points\": [\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ],\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"RECIPROCAL|R-point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"RECIPROCAL|R-corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"RECIPROCAL|R-corrections|s2\",\n      \"branch\": \"RECIPROCAL_POSTERIOR\",\n      \"protocol\": \"R-corrections\",\n      \"point_ids\": [\n        \"P-1\",\n        \"P-2\",\n        \"P-3\",\n        \"P+1\",\n        \"H1\",\n        \"S1\"\n      ],\n      \"points\": [\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ],\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1686\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"RECIPROCAL|R-point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"RECIPROCAL|R-corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+15+0|point\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+15+0\",\n      \"point_ids\": [\n        \"P+1\"\n      ],\n      \"points\": [\n        [\n          2603,\n          1785\n        ]\n      ],\n      \"labels\": [\n        1\n      ],\n      \"box\": null,\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+15+0|point|0\",\n        \"PERTURB_POINT|P+1|d+15+0|point|1\",\n        \"PERTURB_POINT|P+1|d+15+0|point|2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+15+0|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2603,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d+15+0|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+15+0|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+15+0|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2603,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d+15+0|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+15+0|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+15+0|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2603,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d+15+0|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+15+0|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d-15+0|point\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\"\n      ],\n      \"points\": [\n        [\n          2573,\n          1785\n        ]\n      ],\n      \"labels\": [\n        1\n      ],\n      \"box\": null,\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d-15+0|point|0\",\n        \"PERTURB_POINT|P+1|d-15+0|point|1\",\n        \"PERTURB_POINT|P+1|d-15+0|point|2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d-15+0|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2573,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d-15+0|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d-15+0|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d-15+0|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2573,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d-15+0|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d-15+0|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d-15+0|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2573,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d-15+0|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d-15+0|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+0+15|point\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1800\n        ]\n      ],\n      \"labels\": [\n        1\n      ],\n      \"box\": null,\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+0+15|point|0\",\n        \"PERTURB_POINT|P+1|d+0+15|point|1\",\n        \"PERTURB_POINT|P+1|d+0+15|point|2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+0+15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1800\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d+0+15|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+0+15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+0+15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1800\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d+0+15|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+0+15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+0+15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1800\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d+0+15|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+0+15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+0-15|point\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1770\n        ]\n      ],\n      \"labels\": [\n        1\n      ],\n      \"box\": null,\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+0-15|point|0\",\n        \"PERTURB_POINT|P+1|d+0-15|point|1\",\n        \"PERTURB_POINT|P+1|d+0-15|point|2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+0-15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1770\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d+0-15|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+0-15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+0-15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1770\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d+0-15|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+0-15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+0-15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1770\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d+0-15|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+0-15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+15+15|point\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+15+15\",\n      \"point_ids\": [\n        \"P+1\"\n      ],\n      \"points\": [\n        [\n          2603,\n          1800\n        ]\n      ],\n      \"labels\": [\n        1\n      ],\n      \"box\": null,\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+15+15|point|0\",\n        \"PERTURB_POINT|P+1|d+15+15|point|1\",\n        \"PERTURB_POINT|P+1|d+15+15|point|2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+15+15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2603,\n          1800\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d+15+15|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+15+15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+15+15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2603,\n          1800\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d+15+15|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+15+15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+15+15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2603,\n          1800\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d+15+15|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+15+15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+15-15|point\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+15-15\",\n      \"point_ids\": [\n        \"P+1\"\n      ],\n      \"points\": [\n        [\n          2603,\n          1770\n        ]\n      ],\n      \"labels\": [\n        1\n      ],\n      \"box\": null,\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+15-15|point|0\",\n        \"PERTURB_POINT|P+1|d+15-15|point|1\",\n        \"PERTURB_POINT|P+1|d+15-15|point|2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+15-15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2603,\n          1770\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d+15-15|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+15-15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+15-15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2603,\n          1770\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d+15-15|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+15-15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d+15-15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d+15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2603,\n          1770\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d+15-15|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d+15-15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d-15+15|point\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\"\n      ],\n      \"points\": [\n        [\n          2573,\n          1800\n        ]\n      ],\n      \"labels\": [\n        1\n      ],\n      \"box\": null,\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d-15+15|point|0\",\n        \"PERTURB_POINT|P+1|d-15+15|point|1\",\n        \"PERTURB_POINT|P+1|d-15+15|point|2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d-15+15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2573,\n          1800\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d-15+15|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d-15+15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d-15+15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2573,\n          1800\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d-15+15|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d-15+15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d-15+15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2573,\n          1800\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d-15+15|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d-15+15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d-15-15|point\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\"\n      ],\n      \"points\": [\n        [\n          2573,\n          1770\n        ]\n      ],\n      \"labels\": [\n        1\n      ],\n      \"box\": null,\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d-15-15|point|0\",\n        \"PERTURB_POINT|P+1|d-15-15|point|1\",\n        \"PERTURB_POINT|P+1|d-15-15|point|2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d-15-15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2573,\n          1770\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d-15-15|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d-15-15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d-15-15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2573,\n          1770\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d-15-15|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d-15-15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|P+1|d-15-15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"P+1\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2573,\n          1770\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_POINT|P+1|d-15-15|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|P+1|d-15-15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15+0|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15+0|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15+0|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15+0|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15+0|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15+0|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15+0|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15+0|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15+0|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15+0|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15+0|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15+0|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15+0|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15+0|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15+0|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15+0|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15+0|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15+0|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15+0|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15+0|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15+0|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15+0|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15+0|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          592\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15+0|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+0+15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+0+15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+0+15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+0+15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+0+15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+0+15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+0+15|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+0+15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+0+15|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+0+15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+0+15|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+0+15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+0-15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+0-15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+0-15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+0-15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+0-15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+0-15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+0-15|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+0-15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+0-15|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+0-15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+0-15|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+0-15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15+15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15+15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15+15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15+15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15+15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15+15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15+15|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15+15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15+15|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15+15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15+15|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15+15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15-15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15-15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15-15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15-15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15-15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15-15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15-15|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15-15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15-15|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15-15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d+15-15|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d+15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3087,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d+15-15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15+15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15+15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15+15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15+15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15+15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15+15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15+15|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15+15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15+15|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15+15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15+15|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          607\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15+15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15-15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15-15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15-15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15-15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15-15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15-15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15-15|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15-15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15-15|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15-15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|H1|d-15-15|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"H1\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3057,\n          577\n        ],\n        [\n          2230,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|H1|d-15-15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15+0|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15+0|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15+0|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15+0|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15+0|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15+0|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15+0|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15+0|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15+0|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15+0|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15+0|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15+0|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15+0|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15+0|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15+0|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15+0|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15+0|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15+0|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15+0|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15+0|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15+0|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15+0|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15+0|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1686\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15+0|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+0+15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+0+15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+0+15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+0+15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+0+15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+0+15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+0+15|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+0+15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+0+15|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+0+15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+0+15|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+0+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+0+15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+0-15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+0-15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+0-15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+0-15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+0-15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+0-15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+0-15|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+0-15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+0-15|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+0-15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+0-15|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2230,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+0-15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15+15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15+15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15+15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15+15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15+15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15+15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15+15|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15+15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15+15|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15+15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15+15|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15+15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15-15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15-15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15-15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15-15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15-15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15-15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15-15|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15-15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15-15|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15-15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d+15-15|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d+15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2245,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d+15-15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15+15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15+15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15+15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15+15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15+15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15+15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15+15|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15+15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15+15|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15+15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15+15|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15+15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1701\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15+15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15-15|point+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15-15|point+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15-15|point+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15-15|point+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15-15|point+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"point+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": null,\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|point\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15-15|point+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15-15|box+corrections|s0\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15-15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15-15|box+corrections|s1\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15-15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_POINT|S1|d-15-15|box+corrections|s2\",\n      \"branch\": \"PERTURB_POINT\",\n      \"protocol\": \"box+corrections\",\n      \"target\": \"S1\",\n      \"perturbation\": \"d-15-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"H1\",\n        \"S1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          3072,\n          592\n        ],\n        [\n          2215,\n          1671\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        1,\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        300,\n        3500,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"BASE|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_POINT|S1|d-15-15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|T+15+0|box\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box\",\n      \"family\": \"TRANSLATE\",\n      \"perturbation\": \"T+15+0\",\n      \"point_ids\": [],\n      \"points\": [],\n      \"labels\": [],\n      \"box\": [\n        2115,\n        300,\n        3515,\n        2247\n      ],\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"PERTURB_BOX|T+15+0|box|0\",\n        \"PERTURB_BOX|T+15+0|box|1\",\n        \"PERTURB_BOX|T+15+0|box|2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|T+15+0|box+corrections|s0\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box+corrections\",\n      \"family\": \"TRANSLATE\",\n      \"perturbation\": \"T+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2115,\n        300,\n        3515,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_BOX|T+15+0|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_BOX|T+15+0|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|T+15+0|box+corrections|s1\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box+corrections\",\n      \"family\": \"TRANSLATE\",\n      \"perturbation\": \"T+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2115,\n        300,\n        3515,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_BOX|T+15+0|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_BOX|T+15+0|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|T+15+0|box+corrections|s2\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box+corrections\",\n      \"family\": \"TRANSLATE\",\n      \"perturbation\": \"T+15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2115,\n        300,\n        3515,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_BOX|T+15+0|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_BOX|T+15+0|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|T-15+0|box\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box\",\n      \"family\": \"TRANSLATE\",\n      \"perturbation\": \"T-15+0\",\n      \"point_ids\": [],\n      \"points\": [],\n      \"labels\": [],\n      \"box\": [\n        2085,\n        300,\n        3485,\n        2247\n      ],\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"PERTURB_BOX|T-15+0|box|0\",\n        \"PERTURB_BOX|T-15+0|box|1\",\n        \"PERTURB_BOX|T-15+0|box|2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|T-15+0|box+corrections|s0\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box+corrections\",\n      \"family\": \"TRANSLATE\",\n      \"perturbation\": \"T-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2085,\n        300,\n        3485,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_BOX|T-15+0|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_BOX|T-15+0|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|T-15+0|box+corrections|s1\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box+corrections\",\n      \"family\": \"TRANSLATE\",\n      \"perturbation\": \"T-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2085,\n        300,\n        3485,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_BOX|T-15+0|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_BOX|T-15+0|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|T-15+0|box+corrections|s2\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box+corrections\",\n      \"family\": \"TRANSLATE\",\n      \"perturbation\": \"T-15+0\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2085,\n        300,\n        3485,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_BOX|T-15+0|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_BOX|T-15+0|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|T+0-15|box\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box\",\n      \"family\": \"TRANSLATE\",\n      \"perturbation\": \"T+0-15\",\n      \"point_ids\": [],\n      \"points\": [],\n      \"labels\": [],\n      \"box\": [\n        2100,\n        285,\n        3500,\n        2232\n      ],\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"PERTURB_BOX|T+0-15|box|0\",\n        \"PERTURB_BOX|T+0-15|box|1\",\n        \"PERTURB_BOX|T+0-15|box|2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|T+0-15|box+corrections|s0\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box+corrections\",\n      \"family\": \"TRANSLATE\",\n      \"perturbation\": \"T+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        285,\n        3500,\n        2232\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_BOX|T+0-15|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_BOX|T+0-15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|T+0-15|box+corrections|s1\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box+corrections\",\n      \"family\": \"TRANSLATE\",\n      \"perturbation\": \"T+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        285,\n        3500,\n        2232\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_BOX|T+0-15|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_BOX|T+0-15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|T+0-15|box+corrections|s2\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box+corrections\",\n      \"family\": \"TRANSLATE\",\n      \"perturbation\": \"T+0-15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2100,\n        285,\n        3500,\n        2232\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_BOX|T+0-15|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_BOX|T+0-15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|E15|box\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box\",\n      \"family\": \"EXPAND\",\n      \"perturbation\": \"E15\",\n      \"point_ids\": [],\n      \"points\": [],\n      \"labels\": [],\n      \"box\": [\n        2085,\n        285,\n        3515,\n        2247\n      ],\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"PERTURB_BOX|E15|box|0\",\n        \"PERTURB_BOX|E15|box|1\",\n        \"PERTURB_BOX|E15|box|2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|E15|box+corrections|s0\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box+corrections\",\n      \"family\": \"EXPAND\",\n      \"perturbation\": \"E15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2085,\n        285,\n        3515,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_BOX|E15|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_BOX|E15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|E15|box+corrections|s1\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box+corrections\",\n      \"family\": \"EXPAND\",\n      \"perturbation\": \"E15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2085,\n        285,\n        3515,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_BOX|E15|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_BOX|E15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|E15|box+corrections|s2\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box+corrections\",\n      \"family\": \"EXPAND\",\n      \"perturbation\": \"E15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2085,\n        285,\n        3515,\n        2247\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_BOX|E15|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_BOX|E15|box+corrections|s2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|C15|box\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box\",\n      \"family\": \"CONTRACT\",\n      \"perturbation\": \"C15\",\n      \"point_ids\": [],\n      \"points\": [],\n      \"labels\": [],\n      \"box\": [\n        2115,\n        315,\n        3485,\n        2232\n      ],\n      \"mask_input_from\": null,\n      \"multimask_output\": true,\n      \"candidates\": [\n        \"PERTURB_BOX|C15|box|0\",\n        \"PERTURB_BOX|C15|box|1\",\n        \"PERTURB_BOX|C15|box|2\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|C15|box+corrections|s0\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box+corrections\",\n      \"family\": \"CONTRACT\",\n      \"perturbation\": \"C15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2115,\n        315,\n        3485,\n        2232\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_BOX|C15|box\",\n        \"index\": 0\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_BOX|C15|box+corrections|s0\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|C15|box+corrections|s1\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box+corrections\",\n      \"family\": \"CONTRACT\",\n      \"perturbation\": \"C15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2115,\n        315,\n        3485,\n        2232\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_BOX|C15|box\",\n        \"index\": 1\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_BOX|C15|box+corrections|s1\"\n      ]\n    },\n    {\n      \"call_id\": \"PERTURB_BOX|C15|box+corrections|s2\",\n      \"branch\": \"PERTURB_BOX\",\n      \"protocol\": \"box+corrections\",\n      \"family\": \"CONTRACT\",\n      \"perturbation\": \"C15\",\n      \"point_ids\": [\n        \"P+1\",\n        \"P-1\",\n        \"P-2\",\n        \"P-3\"\n      ],\n      \"points\": [\n        [\n          2588,\n          1785\n        ],\n        [\n          2350,\n          900\n        ],\n        [\n          2640,\n          430\n        ],\n        [\n          2400,\n          810\n        ]\n      ],\n      \"labels\": [\n        1,\n        0,\n        0,\n        0\n      ],\n      \"box\": [\n        2115,\n        315,\n        3485,\n        2232\n      ],\n      \"mask_input_from\": {\n        \"call_id\": \"PERTURB_BOX|C15|box\",\n        \"index\": 2\n      },\n      \"multimask_output\": false,\n      \"candidates\": [\n        \"PERTURB_BOX|C15|box+corrections|s2\"\n      ]\n    }\n  ],\n  \"call_plan_semantics\": \"cada llamada es SAM2ImagePredictor.predict(point_coords=points o None, point_labels=labels o None, box=box o None, mask_input=low_res_logits[index][None] de mask_input_from o None, multimask_output, return_logits=True); máscara = logits > 0; mismas conversiones que v1.2 (float32 para puntos y caja, int32 para etiquetas)\",\n  \"analysis_implementation_sha256\": {\n    \"pragma_ae/aem1_v13.py\": \"2e9433604a834db8b5636d112dcd5169b5330727e0a0c05c4e92e9cbc4087da7\",\n    \"pragma_ae/aem1_v13_audit.py\": \"820af07186a171099f94e98541d89d0b0948a3cf4333222c4c0c5a77ebfdc13a\",\n    \"pragma_ae/masks.py\": \"412d1ead02fb92891ca989c5c6ac7a5b2dcd5b6bcdbb57551ec1ce12db712ac9\"\n  },\n  \"consensus\": \"valor de consenso de una celda = el de las dos llaves si coinciden; si no, el de la adjudicación técnica (protocolo v2 rev. 1 §5.2); mientras falte, la candidata no pasa\",\n  \"blind_audit\": {\n    \"protocol\": \"auditoria/PROTOCOLO_AUDITORIA_AEM1_v2.md\",\n    \"protocol_sha256\": \"b5b11c6cce30b534eb83ab9117025e35837e4554eb150e411c8091fec5087a71\",\n    \"candidates\": \"las 18 de +POS_HAIR, +POS_SLEEVE y +POS_HAIR+SLEEVE, más las de BASE que no se reproduzcan bit a bit\",\n    \"not_blind_judged\": [\n      \"RECIPROCAL_POSTERIOR\",\n      \"PERTURB_POINT\",\n      \"PERTURB_BOX\"\n    ],\n    \"order\": \"el paquete va a ChatGPT antes que cualquier resultado; los juicios de Claude se comprometen por hash\"\n  },\n  \"metrics\": {\n    \"perturbation_stability\": {\n      \"function\": \"pragma_ae.aem1_v13.perturbation_stability\",\n      \"unit\": \"cada candidata (protocolo, índice) frente a la misma candidata con cada perturbación válida\",\n      \"labels\": {\n        \"NOT_EVALUABLE\": \"ninguna perturbación válida\",\n        \"CONFLICT\": \"el cribado O* (algún sentinela de la persona posterior filtrado o ninguno) cambia entre base y alguna perturbación\",\n        \"UNSTABLE\": \"IoU global mínimo < 0,90\",\n        \"STABLE\": \"en otro caso\"\n      },\n      \"family_summary\": \"la peor etiqueta evaluable (CONFLICT > UNSTABLE > STABLE)\",\n      \"secondary\": \"IoU mínimo dentro de CONTACT_BOX (None si las dos están vacías en la caja)\",\n      \"continuous\": \"cobertura de cada O* (base y cada perturbación) y, en cada cruce, su distancia al umbral 0,20 (ChatGPT 003 c)\",\n      \"base_of_each_family\": {\n        \"P+1\": \"BASE (point y point+corrections)\",\n        \"H1 y S1\": \"+POS_HAIR+SLEEVE\",\n        \"caja\": \"BASE (box y box+corrections)\"\n      }\n    },\n    \"reciprocal_stability\": {\n      \"function\": \"pragma_ae.aem1_v13.reciprocal_stability\",\n      \"unit\": \"las 3 semillas R-corrections, IoU por pares dentro de CONTACT_BOX\",\n      \"labels\": {\n        \"NOT_EVALUABLE\": \"alguna vacía en la caja\",\n        \"CONFLICT\": \"el cribado K* (la chica filtrada dentro de R) cambia entre semillas\",\n        \"UNSTABLE\": \"IoU mínimo < 0,90\",\n        \"STABLE\": \"en otro caso\"\n      }\n    },\n    \"ownership\": {\n      \"function\": \"pragma_ae.aem1_v13.ownership\",\n      \"unit\": \"cada candidata de la chica (BASE y +POS) frente a R_ref, dentro de CONTACT_BOX\",\n      \"ratio\": \"|T∩R| / min(|T|, |R|) en la caja; se reportan siempre también |T∩R|/|T| y |T∩R|/|R|\",\n      \"role\": \"diagnóstico, nunca criterio de aceptación (ChatGPT 003 d)\",\n      \"labels\": {\n        \"NOT_EVALUABLE\": \"T o R vacía en la caja\",\n        \"DISJOINT\": \"≤ 0,02\",\n        \"MARGINAL\": \"entre 0,02 y 0,10\",\n        \"SHARED\": \"≥ 0,10\"\n      },\n      \"also_reported\": [\n        \"T filtra O2 u O3 (> 0,20)\",\n        \"R_ref cubre O2 y O3 (≥ 0,80)\"\n      ]\n    },\n    \"interpretation\": {\n      \"function\": \"pragma_ae.aem1_v13.interpret_reciprocal\",\n      \"table\": {\n        \"NO_INFERENCE\": \"R no es STABLE o la propiedad no es evaluable\",\n        \"OWNERSHIP_AMBIGUOUS\": \"T arrastra el moño y T, R lo reclaman a la vez (SHARED)\",\n        \"BUN_ATTRIBUTED_TO_TARGET_BY_BOTH\": \"T arrastra el moño, T y R son DISJOINT y R no cubre el moño\",\n        \"SEPARATION_CONSISTENT_BOTH_WAYS\": \"T no arrastra el moño y T, R son DISJOINT\",\n        \"MIXED\": \"cualquier otro caso\"\n      },\n      \"scope\": \"diagnóstico del prompting; nunca rechaza SAM 2\"\n    }\n  },\n  \"hypotheses\": [\n    {\n      \"id\": \"H-C1\",\n      \"by\": \"Claude (carta 002)\",\n      \"function\": \"pragma_ae.aem1_v13.hypothesis_h_c1\",\n      \"statement\": \"+POS_HAIR recupera el pelo de la chica pero vuelve a arrastrar el moño\",\n      \"unit\": \"las 6 candidatas de +POS_HAIR\",\n      \"holds_if\": \"≥ 4 con target_hair_included TRUE en ambas llaves y (other_person_excluded FALSE en ambas llaves u O2/O3 filtrado)\",\n      \"refuted_if\": \"≥ 4 con target_hair_included TRUE en ambas llaves, other_person_excluded TRUE en ambas y sin O2/O3 filtrado\",\n      \"otherwise\": \"INDETERMINATE\",\n      \"inference_limit\": \"aunque se cumpla, solo muestra que esta familia de prompts no resuelve la ambigüedad (ChatGPT 002)\"\n    },\n    {\n      \"id\": \"H-C2\",\n      \"by\": \"Claude (carta 003)\",\n      \"function\": \"pragma_ae.aem1_v13.hypothesis_h_c2\",\n      \"statement\": \"PERTURB_POINT sobre P+1 no es STABLE en el protocolo point\",\n      \"reason\": \"P+1 (2588, 1785) está en la fila superior de la caja del botón que devolvió point#0 en la corrida 1 (2555–2604 × 1785–1824)\",\n      \"holds_if\": \"alguna de las 3 salidas de point es UNSTABLE o CONFLICT\",\n      \"refuted_if\": \"las 3 salidas de point son STABLE\"\n    },\n    {\n      \"id\": \"H-G1\",\n      \"by\": \"ChatGPT (carta 003)\",\n      \"function\": \"pragma_ae.aem1_v13.hypothesis_h_g1\",\n      \"statement\": \"S1 tendrá un efecto predominantemente local sobre la recuperación de mangas y no sobre la propiedad del moño\",\n      \"unit\": \"las 6 candidatas de +POS_SLEEVE, cada una frente a su BASE emparejada (mismo protocolo y semilla)\",\n      \"holds_if\": \"≥ 4/6 con target_dark_sleeves_included TRUE en ambas llaves y ≤ 2/6 empeoran other_person_excluded frente a su BASE\",\n      \"refuted_if\": \"≥ 4 no recuperan mangas (no TRUE en ambas llaves) o ≥ 4 introducen una fuga nueva de la persona posterior\",\n      \"otherwise\": \"INDETERMINATE\",\n      \"operationalization_by_claude\": \"«empeora» = BASE emparejada con other_person_excluded TRUE (referencia si es bit a bit; si no, su consenso ciego) y +POS_SLEEVE con consenso FALSE\"\n    },\n    {\n      \"id\": \"H-G2\",\n      \"by\": \"ChatGPT (carta 003)\",\n      \"function\": \"pragma_ae.aem1_v13.hypothesis_h_g2\",\n      \"statement\": \"las correcciones recíprocas no cambiarán de propietario grueso entre semillas\",\n      \"holds_if\": \"reciprocal_stability ∈ {STABLE, UNSTABLE}\",\n      \"refuted_if\": \"reciprocal_stability = CONFLICT\",\n      \"otherwise\": \"NOT_EVALUABLE no confirma ni refuta (INDETERMINATE)\",\n      \"reason\": \"tres anclas positivas sobre la posterior y negativos explícitos sobre la chica deberían estabilizar qué persona se elige antes que su frontera\"\n    },\n    {\n      \"id\": \"H-G3\",\n      \"by\": \"ChatGPT (carta 003)\",\n      \"function\": \"pragma_ae.aem1_v13.hypothesis_h_g3\",\n      \"statement\": \"al menos una T que arrastre O2/O3 será SHARED con R_ref mientras R también cubre O2/O3\",\n      \"unit\": \"candidatas de la chica de BASE y +POS\",\n      \"holds_if\": \"existe T con fuga O2/O3, R_ref cubre O2/O3 y la propiedad es SHARED\",\n      \"refuted_if\": \"R_ref cubre O2/O3 y todas las T con fuga O2/O3 son DISJOINT (y hay al menos una)\",\n      \"otherwise\": \"INDETERMINATE (también si R no es evaluable)\",\n      \"operationalization_by_claude\": \"«fuga O2/O3» = cobertura de O2 u O3 > 0,20; «R cubre O2/O3» = O2 y O3 ≥ 0,80 en R_ref\"\n    },\n    {\n      \"id\": \"H-G4\",\n      \"by\": \"ChatGPT (carta 003)\",\n      \"function\": \"pragma_ae.aem1_v13.hypothesis_h_g4\",\n      \"statement\": \"+POS_HAIR+SLEEVE conseguirá al menos una candidata que recupere a la vez pelo y mangas\",\n      \"holds_if\": \"≥ 1/6 con target_hair_included y target_dark_sleeves_included TRUE en ambas llaves\",\n      \"refuted_if\": \"0/6\",\n      \"note\": \"no predice que eso baste para un PASS de separación\"\n    }\n  ],\n  \"decision\": {\n    \"acceptance\": \"solo por el protocolo de auditoría v2 rev. 1 §5 (doble llave y adjudicación técnica); ninguna métrica de este archivo acepta ni rechaza\",\n    \"user_veto\": true,\n    \"sam2_rejectable\": false,\n    \"project_status\": \"INCONCLUSIVE_A_E0_REQUIRED\",\n    \"phase_b\": \"BLOQUEADA\"\n  },\n  \"forbidden\": [\n    \"reparar la máscara de la chica con la del prompt recíproco (objetivo − posterior)\",\n    \"elegir semilla, salida o candidata por score\",\n    \"combinar ramas (no hay combinación prerregistrada)\",\n    \"cambiar prompts, caja o umbrales después de ver datos de la corrida\",\n    \"ejecutar un caso INVALID_PERTURBATION\",\n    \"FastAPI, localhost, YOLO-seg o BiRefNet; tocar la extensión\"\n  ],\n  \"outputs\": {\n    \"zip\": \"PRAGMA_AEM1v13_<run_id>_PENDING_EXTERNAL_AUDIT.zip\",\n    \"contents\": [\n      \"aem1v13_config.json (prerregistro embebido, entorno, compuertas y luma)\",\n      \"aem1v13_calls.json (las llamadas ejecutadas, en orden)\",\n      \"aem1v13_manifest.json (bytes y SHA-256)\",\n      \"aem1v13_report.json\",\n      \"masks/<candidata>.png para BASE, +POS y RECIPROCAL (36)\",\n      \"aem1v13_perturbaciones.npz (174 máscaras en packbits, con SHA-256 por máscara en el manifiesto)\"\n    ],\n    \"not_shown_in_colab\": \"el cuaderno no muestra máscaras, áreas, scores ni sentinelas: solo progreso e integridad\"\n  },\n  \"content_sha256\": \"5800f2bf624620c353067210c73d782a8aefa39563925e859929ef0cba432a8c\"\n}\n"
assert hashlib.sha256(PREREG_TEXT.encode("utf-8")).hexdigest() == PREREG_FILE_SHA256, "Prerregistro alterado"
PREREG = json.loads(PREREG_TEXT)
CALL_PLAN = PREREG["call_plan"]
assert PREREG["status"] == "PREREGISTERED" and len(CALL_PLAN) == PREREG["candidate_counts"]["calls"]

gray = image.astype(np.float32).mean(axis=2)   # misma luma que el preflight: media de canales, parche 13×13

def patch_luma(xy, radius=6):
    x, y = int(xy[0]), int(xy[1])
    return float(gray[max(0, y - radius):y + radius + 1, max(0, x - radius):x + radius + 1].mean())

expected_luma = {}
for entry in PREREG["base_config"]["prompts"]:
    expected_luma[tuple(entry["xy"])] = entry["patch_luma_mean"]
for group in PREREG["base_config"]["holdouts"].values():
    for entry in group:
        expected_luma[tuple(entry["xy"])] = entry["patch_luma_mean"]
for entry in PREREG["new_prompts"].values():
    expected_luma[tuple(entry["xy"])] = entry["patch_luma_mean"]
for target in PREREG["branches"]["PERTURB_POINT"]["targets"].values():
    for row in target["perturbations"]:
        expected_luma[tuple(row["xy"])] = row["patch_luma_mean"]
used = {tuple(p) for call in CALL_PLAN for p in call["points"]}
missing = sorted(used - set(expected_luma))
if missing:
    raise RuntimeError(f"FAIL_CONFIG: prompts sin luma registrada: {missing[:5]}")
deviations = {xy: abs(patch_luma(xy) - value) for xy, value in expected_luma.items()}
LUMA_CHECK = {"points_checked": len(deviations), "max_abs_deviation": round(max(deviations.values()), 3), "tolerance": 3.0}
if LUMA_CHECK["max_abs_deviation"] > 3.0:
    raise RuntimeError(f"FAIL_CONFIG: la luma se desvía {LUMA_CHECK['max_abs_deviation']} > 3,0. No se genera nada.")
print("Prerregistro verificado:", PREREG["content_sha256"][:12], "…", "·", len(CALL_PLAN), "llamadas ·",
      "luma de", LUMA_CHECK["points_checked"], "puntos dentro de 3,0")

In [ ]:
# Celda 4 · modelo y un único embedding de la foto (las mismas llamadas que v1.2)
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

MASK_THRESHOLD = 0.0
t0 = time.perf_counter()
try:
    sam2_model = build_sam2(MODEL_CFG, str(CHECKPOINT), device=DEVICE)
    predictor = SAM2ImagePredictor(sam2_model)
    synchronize()
except torch.cuda.OutOfMemoryError as exc:
    raise RuntimeError("FAIL_ENVIRONMENT: Large no cabe; usa L4 o A100. No se degrada a Small.") from exc
MODEL_LOAD_S = time.perf_counter() - t0

@dataclass
class PromptSet:
    points: np.ndarray
    labels: np.ndarray
    def __post_init__(self):
        self.points = np.asarray(self.points, dtype=np.float32).reshape(-1, 2)
        self.labels = np.asarray(self.labels, dtype=np.int32).reshape(-1)

def generate(prompts=None, box=None, mask_input=None, multimask_output=True):
    synchronize(); start = time.perf_counter()
    with torch.inference_mode(), inference_precision():
        hr_logits, scores, low_res = predictor.predict(
            point_coords=prompts.points if prompts is not None else None,
            point_labels=prompts.labels if prompts is not None else None,
            box=box, mask_input=mask_input, multimask_output=multimask_output, return_logits=True)
    synchronize()
    return (np.asarray(hr_logits, dtype=np.float32) > MASK_THRESHOLD, np.asarray(scores, dtype=np.float32),
            np.asarray(low_res, dtype=np.float32), time.perf_counter() - start)

synchronize(); t0 = time.perf_counter()
with torch.inference_mode(), inference_precision():
    predictor.set_image(image)
synchronize()
EMBEDDING_S = time.perf_counter() - t0
print(f"Modelo listo en {MODEL_LOAD_S:.1f} s · embedding en {EMBEDDING_S:.2f} s")

In [ ]:
# Celda 5 · las 178 llamadas del prerregistro, en orden. Solo se imprime el progreso.
PNG_BRANCHES = ("BASE", "+POS_HAIR", "+POS_SLEEVE", "+POS_HAIR+SLEEVE", "RECIPROCAL")
LOW_RES, EXECUTED, PNG_MASKS, PACKED, PACKED_SHA = {}, [], {}, {}, {}
t_run = time.perf_counter()
for number, call in enumerate(CALL_PLAN, 1):
    prompts = PromptSet(call["points"], call["labels"]) if call["points"] else None
    box = None if call["box"] is None else np.asarray(call["box"], np.float32)
    mask_input = None
    if call["mask_input_from"] is not None:
        source = call["mask_input_from"]
        mask_input = LOW_RES[source["call_id"]][source["index"]][None, :, :]
    masks, scores, low_res, seconds = generate(prompts, box, mask_input, call["multimask_output"])
    assert masks.shape == (len(call["candidates"]), 2248, 4000), masks.shape
    if call["multimask_output"]:
        LOW_RES[call["call_id"]] = low_res          # solo se reutilizan como semillas
    for cid, mask in zip(call["candidates"], masks):
        packed = np.packbits(mask)
        PACKED_SHA[cid] = hashlib.sha256(packed.tobytes()).hexdigest()
        if cid.split("|", 1)[0] in PNG_BRANCHES:
            PNG_MASKS[cid] = mask
        else:
            PACKED[cid] = packed
    EXECUTED.append({**{k: call[k] for k in ("call_id", "branch", "protocol", "points", "labels", "box",
                                               "mask_input_from", "multimask_output", "candidates")},
                     "scores_never_used": [round(float(s), 6) for s in scores], "inference_s": round(seconds, 4)})
    if number % 20 == 0 or number == len(CALL_PLAN):
        print(f"llamada {number}/{len(CALL_PLAN)}")
RUN_S = time.perf_counter() - t_run
assert len(PACKED_SHA) == PREREG["candidate_counts"]["total_masks"]
print(f"Hecho: {len(PACKED_SHA)} máscaras en {RUN_S:.1f} s. No se muestran: la auditoría es ciega.")

In [ ]:
# Celda 6 · manifiesto, ZIP y descarga
def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return path

(RUN_DIR / "masks").mkdir(exist_ok=True)
for cid, mask in PNG_MASKS.items():
    Image.fromarray(mask.astype(np.uint8) * 255).save(RUN_DIR / "masks" / (cid.replace("|", "__") + ".png"))
np.savez_compressed(RUN_DIR / "aem1v13_perturbaciones.npz", __shape__=np.array([2248, 4000]),
                    **{cid.replace("|", "__"): packed for cid, packed in PACKED.items()})
write_json(RUN_DIR / "aem1v13_config.json", {
    "notebook_version": "1.3", "run_id": RUN_ID, "prereg_content_sha256": PREREG["content_sha256"],
    "prereg_file_sha256": PREREG_FILE_SHA256, "prereg": PREREG, "environment": ENVIRONMENT, "luma_check": LUMA_CHECK,
    "gates": {"sam2_commit": "verificado en la celda 1", "checkpoint": "verificado en la celda 1",
              "image_sha256": "verificado en la celda 2", "prereg": "verificado en la celda 3", "luma": LUMA_CHECK},
})
write_json(RUN_DIR / "aem1v13_calls.json", EXECUTED)
write_json(RUN_DIR / "aem1v13_report.json", {
    "run_id": RUN_ID, "status": "PENDING_EXTERNAL_AUDIT", "environment": ENVIRONMENT,
    "timings_s": {"model_load": round(MODEL_LOAD_S, 3), "embedding": round(EMBEDDING_S, 3), "calls": round(RUN_S, 3)},
    "gpu_peak_gib": round(torch.cuda.max_memory_allocated() / 2**30, 3) if DEVICE == "cuda" else None,
    "counts": {"calls": len(EXECUTED), "masks": len(PACKED_SHA), "png": len(PNG_MASKS), "npz": len(PACKED)},
    "note": "sin revisión en Colab: auditoría ciega externa según auditoria/PROTOCOLO_AUDITORIA_AEM1_v2.md (rev. 1)",
})
files_listed = sorted(p for p in RUN_DIR.rglob("*") if p.is_file() and p.name != "P1070614.JPG"
                      and p.name != "aem1v13_manifest.json")
manifest = {"run_id": RUN_ID, "files": {p.relative_to(RUN_DIR).as_posix(): {"bytes": p.stat().st_size, "sha256": sha256_file(p)}
                                        for p in files_listed},
            "masks": dict(sorted(PACKED_SHA.items())), "mask_hash": "sha256(np.packbits(mask)) sin cabecera"}
write_json(RUN_DIR / "aem1v13_manifest.json", manifest)
ZIP_PATH = WORK_DIR / f"PRAGMA_AEM1v13_{RUN_ID}_PENDING_EXTERNAL_AUDIT.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for p in files_listed + [RUN_DIR / "aem1v13_manifest.json"]:
        archive.write(p, p.relative_to(RUN_DIR).as_posix())
with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
ZIP_SHA256 = sha256_file(ZIP_PATH)
print(f"ZIP: {ZIP_PATH.name} · {ZIP_PATH.stat().st_size / 2**20:.1f} MiB · SHA-256 {ZIP_SHA256}")
if not PRAGMA_HEADLESS:
    files.download(str(ZIP_PATH))
print("Adjunta este ZIP solo a Claude. No compartas capturas de esta corrida.")

## 7. Qué hacer ahora

1. Busca el ZIP `PRAGMA_AEM1v13_…_PENDING_EXTERNAL_AUDIT.zip` en tu carpeta de descargas.
2. **Adjúntalo a Claude** en la sesión de Claude Code. No se lo mandes a ChatGPT.
3. Claude verifica la integridad sin mirar resultados, prepara el paquete ciego y te lo da. Ese
   paquete es lo único que recibe ChatGPT, sin carta.